# Python Scripting — Assignment Answers
**Course:** Python / DSA — PW Skills  
**Topic:** Python Scripting

This notebook covers two scripting tasks:
1. **Basic file operations and data processing**
2. **A simple web scraper to extract data from a website**

Each section includes a short explanation followed by working, runnable code.

---

## 📁 Question 1 — Basic File Operations and Data Processing

**Goal:** demonstrate Python's built-in capabilities for working with files and processing the data inside them.

We will cover:
- Writing data to a text file  
- Reading data from a text file  
- Appending new content  
- Working with a **CSV file** (creating, reading, filtering, aggregating)  
- Basic **data processing** — counting, averaging, sorting

> All files are created inside the Colab session at runtime so the notebook is self-contained — no external uploads required.

### 1.1 — Writing and Reading a Text File

In [ ]:
# Write some content to a text file
with open("notes.txt", "w") as f:
    f.write("Python is powerful.\n")
    f.write("File handling is essential.\n")
    f.write("Scripting saves time.\n")

# Read the entire file
with open("notes.txt", "r") as f:
    content = f.read()

print("---- File Content ----")
print(content)


### 1.2 — Appending Data and Reading Line by Line

In [ ]:
# Append a new line
with open("notes.txt", "a") as f:
    f.write("Always close files properly.\n")

# Read line by line
print("---- Line by Line ----")
with open("notes.txt", "r") as f:
    for i, line in enumerate(f, start=1):
        print(f"Line {i}: {line.strip()}")


### 1.3 — Creating a CSV File (Sample Sales Data)

We will create a CSV of product sales, then process it.

In [ ]:
import csv

rows = [
    ["product",  "category",   "quantity", "price"],
    ["Laptop",   "Electronics", 3,         55000],
    ["Mouse",    "Electronics", 15,        450],
    ["Notebook", "Stationery",  40,        60],
    ["Pen",      "Stationery",  100,       10],
    ["Headphone","Electronics", 8,         1200],
    ["Eraser",   "Stationery",  60,        5],
    ["Monitor",  "Electronics", 4,         12000],
]

with open("sales.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print("sales.csv created successfully.")


### 1.4 — Reading the CSV File

In [ ]:
import csv

with open("sales.csv", "r") as f:
    reader = csv.DictReader(f)
    data = list(reader)

print("---- Raw rows ----")
for row in data:
    print(row)


### 1.5 — Data Processing on the CSV

We compute:
- Total revenue per product  
- Total quantity sold  
- Average price per category  
- Top-selling product by revenue

In [ ]:
# Convert numeric fields
for row in data:
    row["quantity"] = int(row["quantity"])
    row["price"]    = float(row["price"])
    row["revenue"]  = row["quantity"] * row["price"]

# 1) Total revenue per product
print("---- Revenue per product ----")
for row in data:
    print(f"{row['product']:<10}  ₹ {row['revenue']:,.2f}")

# 2) Total quantity sold across all products
total_qty = sum(r["quantity"] for r in data)
print(f"\nTotal items sold : {total_qty}")

# 3) Total revenue overall
total_rev = sum(r["revenue"] for r in data)
print(f"Total revenue    : ₹ {total_rev:,.2f}")

# 4) Average price per category
from collections import defaultdict
cat_prices = defaultdict(list)
for r in data:
    cat_prices[r["category"]].append(r["price"])

print("\n---- Average price per category ----")
for cat, prices in cat_prices.items():
    print(f"{cat:<12}  avg ₹ {sum(prices)/len(prices):,.2f}")

# 5) Top product by revenue
top = max(data, key=lambda r: r["revenue"])
print(f"\n🏆 Top product by revenue: {top['product']}  (₹ {top['revenue']:,.2f})")


### 1.6 — Writing the Processed Result to a New CSV

In [ ]:
import csv

# Sort products by revenue (descending) and save to a new file
data_sorted = sorted(data, key=lambda r: r["revenue"], reverse=True)

with open("sales_report.csv", "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["product", "category", "quantity", "price", "revenue"]
    )
    writer.writeheader()
    writer.writerows(data_sorted)

# Verify
print("---- sales_report.csv ----")
with open("sales_report.csv", "r") as f:
    print(f.read())


---
## 🌐 Question 2 — Simple Web Scraper

**Goal:** build a scraper that extracts structured data from a website.

We will scrape [`http://quotes.toscrape.com`](http://quotes.toscrape.com) — a public website explicitly designed for practicing web scraping. We extract each quote's:
- Quote text
- Author
- Tags

We will then save the results to a CSV file.

**Libraries used:**
- `requests` — fetches the HTML page
- `BeautifulSoup` (from `bs4`) — parses HTML and lets us extract elements with CSS selectors

> Both libraries come pre-installed on Google Colab. If you run this locally and they're missing, the install cell below will handle it.

### 2.1 — Install Dependencies (only if needed)

In [ ]:
# Colab already has these; uncomment if you run locally and need to install
# !pip install requests beautifulsoup4


### 2.2 — Scrape a Single Page

In [ ]:
import requests
from bs4 import BeautifulSoup

URL = "http://quotes.toscrape.com/page/1/"
headers = {"User-Agent": "Mozilla/5.0 (compatible; PythonScraper/1.0)"}

response = requests.get(URL, headers=headers, timeout=10)
print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")
quote_blocks = soup.find_all("div", class_="quote")

print(f"Found {len(quote_blocks)} quotes on page 1.\n")

# Show the first one as a preview
first = quote_blocks[0]
print("Quote :", first.find("span", class_="text").get_text(strip=True))
print("Author:", first.find("small", class_="author").get_text(strip=True))
print("Tags  :", [t.get_text(strip=True) for t in first.find_all("a", class_="tag")])


### 2.3 — Scrape All Pages (Pagination)

The site has multiple pages. We loop until no "Next" button is found, gathering every quote.

In [ ]:
import requests
from bs4 import BeautifulSoup

BASE = "http://quotes.toscrape.com"
headers = {"User-Agent": "Mozilla/5.0 (compatible; PythonScraper/1.0)"}

all_quotes = []
url = BASE + "/page/1/"
page_num = 1

while url:
    print(f"Scraping page {page_num} ...")
    r = requests.get(url, headers=headers, timeout=10)
    if r.status_code != 200:
        print(f"  Failed with status {r.status_code}, stopping.")
        break

    soup = BeautifulSoup(r.text, "html.parser")

    for block in soup.find_all("div", class_="quote"):
        all_quotes.append({
            "text":   block.find("span", class_="text").get_text(strip=True),
            "author": block.find("small", class_="author").get_text(strip=True),
            "tags":   ", ".join(t.get_text(strip=True)
                                for t in block.find_all("a", class_="tag")),
        })

    # Follow "Next" link if present
    next_btn = soup.find("li", class_="next")
    url = BASE + next_btn.a["href"] if next_btn else None
    page_num += 1

print(f"\n✅ Total quotes scraped: {len(all_quotes)}")

# Preview first 3
for q in all_quotes[:3]:
    print("\n-", q["text"][:80] + ("..." if len(q["text"]) > 80 else ""))
    print("  —", q["author"], "| tags:", q["tags"])


### 2.4 — Save Scraped Data to a CSV File

In [ ]:
import csv

with open("quotes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["text", "author", "tags"])
    writer.writeheader()
    writer.writerows(all_quotes)

print(f"Saved {len(all_quotes)} quotes to quotes.csv")

# Quick check — first 500 chars of the file
with open("quotes.csv", "r", encoding="utf-8") as f:
    print("\n---- Preview of quotes.csv ----")
    print(f.read()[:500])


### 2.5 — Bonus: Quick Analysis on the Scraped Data

A small example of combining scraping with data processing — count quotes per author.

In [ ]:
from collections import Counter

author_counts = Counter(q["author"] for q in all_quotes)

print("---- Top 5 authors by quote count ----")
for author, count in author_counts.most_common(5):
    print(f"{author:<25}  {count} quote(s)")


---
## ✅ End of Assignment

Both scripting tasks are complete:

1. **File operations & data processing** — text + CSV read/write, append, aggregation, sorting, and saving processed output.  
2. **Web scraper** — paginated scrape of `quotes.toscrape.com`, saved to CSV, with a small bonus analysis.

> ⚠️ **Ethics note on scraping:** Always check a site's `robots.txt` and Terms of Service before scraping. `quotes.toscrape.com` is explicitly created for practice, which is why it's used here.